# Reproduce the paper’s paired-bootstrap tables

This HSC recipe reads saved reference predictions and catalogs without retraining. Outputs are written only to a separate output directory. Point metrics use the 15% development holdout; reliability metrics use the independent 7.5% final-test cohort.


In [ ]:
LEGACY_PROJECT_ROOT = ""  # required: trusted reference artifact/data workspace
OUTPUT_DIR = "runs/paper-bootstrap"
N_BOOT = 2000
CI_LEVEL = 0.95
BOOTSTRAP_SEED = 20260917
N_JOBS = 1


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
from turbopdz.recipes.hsc_bootstrap import run_manifest, make_latex_tables, claim_summary
from turbopdz.artifacts import write_json

checkout = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
if not LEGACY_PROJECT_ROOT:
    raise ValueError("Set LEGACY_PROJECT_ROOT; private catalog data are not bundled.")
source = Path(LEGACY_PROJECT_ROOT).resolve()
output = (checkout / OUTPUT_DIR).resolve()
if output.is_relative_to(source):
    raise ValueError("Choose an output directory outside the legacy workspace.")
manifest = json.loads((checkout / "recipes/hsc-pdr3/paper-runs.json").read_text())
results = run_manifest(source, manifest, n_boot=N_BOOT, bootstrap_seed=BOOTSTRAP_SEED,
                       ci_level=CI_LEVEL, retained_grid=np.linspace(0.05, 1, 100), n_jobs=N_JOBS)


In [ ]:
output.mkdir(parents=True, exist_ok=True)
summary = pd.concat([item["summary"] for item in results], ignore_index=True)
summary.to_csv(output / "paired_bootstrap_summary.csv", index=False)
claims = claim_summary(summary)
claims.to_csv(output / "paired_bootstrap_claim_counts.csv", index=False)
for filename, contents in make_latex_tables(summary).items():
    (output / filename).write_text(contents)
for spec, result in zip(manifest, results):
    np.savez_compressed(output / f"{spec['cat']}_{spec['layer']}_draws.npz", **result["draws"])
write_json(output / "provenance.json", {"manifest": manifest, "n_boot": N_BOOT,
                                      "ci_level": CI_LEVEL, "bootstrap_seed": BOOTSTRAP_SEED,
                                      "validated_runs": [item["provenance"] for item in results]})
display(claims)
